# FIDB sensitivity study

**How controlled build changes affect Ghidra Function ID hashes**

The results below are to be considered indicative only at the time of writing.

Our key questions are:

- Which compilation factors change FID output, and by how much?
- Which recovery settings produce the most usable FID hashes?
- How can false positives be minimised?

## Part 1 — Build-factor sensitivity

Every row is a within-route comparison of the same pinned library source under one declared build change.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_project_root() -> Path:
    working_tree = [Path.cwd(), *Path.cwd().parents]
    candidates = []
    for path in working_tree:
        candidates.extend(
            [
                path,
                path / "FIDB_sensetivity",
                path / "FIDB_sensitivity",
                path / "FIDB_Sensetivity",
            ]
        )
    for path in candidates:
        if (path / "data/sensitivity_summary.csv").is_file():
            return path
    raise FileNotFoundError(
        "Could not find data/sensitivity_summary.csv beside the notebook "
        "or in a FIDB_sensetivity project folder."
    )


ROOT = find_project_root()
DATA = ROOT / "data"
summary = pd.read_csv(DATA / "sensitivity_summary.csv")
recovery = pd.read_csv(DATA / "ghidra_recovery_summary.csv")

In [2]:
def show_percent(value):
    return "—" if pd.isna(value) else f"{100 * value:.2f}%"


def show_count(value):
    return "—" if pd.isna(value) else f"{int(value):,}"


def show_fraction(count, total):
    if not total:
        return "—"
    return f"{100 * count / total:.2f}% ({int(count):,}/{int(total):,})"


def show_hash_retention(row, count_column, conditional_column):
    total = row["hash_truth_pairs"]
    if not total:
        return "—"
    overall = f'{100 * row[count_column] / total:.2f}% of tracked'
    conditional = show_percent(row[conditional_column])
    return f"{overall}; {conditional} of hashable"


headline = pd.DataFrame(
    {
        "Route": summary["route"],
        "Factor": summary["factor"],
        "Controlled comparison": summary["comparison"],
        "Tracked functions": summary["hash_truth_pairs"].map(show_count),
        "Available in both analyses": summary.apply(
            lambda row: show_fraction(
                row["discovery_retained_n"], row["hash_truth_pairs"]
            ),
            axis=1,
        ),
        "Hashable in both analyses": summary.apply(
            lambda row: show_fraction(row["comparable_pairs"], row["hash_truth_pairs"]),
            axis=1,
        ),
        "Same full hash": summary.apply(
            lambda row: show_hash_retention(
                row, "full_hash_same_n", "full_hash_same_rate"
            ),
            axis=1,
        ),
        "Same specific hash": summary.apply(
            lambda row: show_hash_retention(
                row, "specific_hash_same_n", "specific_hash_same_rate"
            ),
            axis=1,
        ),
        "Reference-matrix action": summary["matrix_action"],
    }
)

display(
    headline.style.hide(axis="index")
    .set_properties(**{"text-align": "left", "white-space": "normal"})
    .set_table_styles(
        [
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "table", "props": [("font-size", "12px")]},
        ]
    )
)

Route,Factor,Controlled comparison,Tracked functions,Available in both analyses,Hashable in both analyses,Same full hash,Same specific hash,Reference-matrix action
macOS ARM64,AddressSanitizer,off to on,"14,871","15.20% (2,260/14,871)","12.63% (1,878/14,871)",0.91% of tracked; 7.24% of hashable,0.51% of tracked; 4.05% of hashable,Do not admit unguarded; wrong label observed
macOS ARM64,Analysis repeat,"same bytes, fresh Ghidra project","5,642","86.71% (4,892/5,642)","70.72% (3,990/5,642)",70.72% of tracked; 100.00% of hashable,70.72% of tracked; 100.00% of hashable,Control: lock the analysis settings
macOS ARM64,Compiler version,Apple Clang 21 to Clang 22,"2,849","79.50% (2,265/2,849)","66.09% (1,883/2,849)",54.48% of tracked; 82.42% of hashable,30.26% of tracked; 45.78% of hashable,Add compiler-version candidate
macOS ARM64,Dead-code elimination,linker dead strip off to on,"2,821","0.00% (0/2,821)","0.00% (0/2,821)",0.00% of tracked; — of hashable,0.00% of tracked; — of hashable,Redesign truth; no attributable pair
macOS ARM64,Debug information,-g to -g0,"2,821","86.71% (2,446/2,821)","70.68% (1,994/2,821)",70.65% of tracked; 99.95% of hashable,70.33% of tracked; 99.50% of hashable,No extra reference on this route
macOS ARM64,Build repeat,same inputs and flags,"5,642","86.71% (4,892/5,642)","70.72% (3,990/5,642)",70.72% of tracked; 100.00% of hashable,70.72% of tracked; 100.00% of hashable,Control: no extra reference
macOS ARM64,Link-time optimisation,off to on,"2,821","0.00% (0/2,821)","0.00% (0/2,821)",0.00% of tracked; — of hashable,0.00% of tracked; — of hashable,Redesign truth; no attributable pair
macOS ARM64,Frame pointer,kept to omitted,"4,305","56.82% (2,446/4,305)","46.34% (1,995/4,305)",14.68% of tracked; 31.68% of hashable,11.68% of tracked; 25.21% of hashable,Add frame-pointer candidate
macOS ARM64,Optimisation,O2 to O0,"3,656","61.95% (2,265/3,656)","51.50% (1,883/3,656)",3.25% of tracked; 6.32% of hashable,3.25% of tracked; 6.32% of hashable,Add optimisation candidate
macOS ARM64,Optimisation,O2 to O1,"2,887","84.41% (2,437/2,887)","68.58% (1,980/2,887)",36.99% of tracked; 53.94% of hashable,21.13% of tracked; 30.81% of hashable,Add optimisation candidate


### Column explainer

- **Tracked functions:** distinct source-function identities independently mapped across the two builds.
- **Available in both analyses:** functions that remained distinct in both builds and were recovered by Ghidra in both. Loss here can come from compilation or analysis.
- **Hashable in both analyses:** available functions for which Ghidra produced usable hashes in both builds.
- **Same full/specific hash:** the first percentage uses every tracked function as the denominator; the second isolates hash churn among the hashable survivors.

Full and specific hashes are stages of the same FID lookup, not separate FID products. Final matches also depend on function discovery, parent/child relations, the score threshold, database contents and ambiguity handling.

### Pre-hash loss

FID cannot ablate a function that is no longer a distinct function, is not recovered by Ghidra, or produces no usable hash. Conditional hash stability therefore describes only the survivors and is optimistic if read alone.

### Ghidra loss

The identical-binary controls separate this from build churn. On macOS, Ghidra did not recover **750** mapped truth functions and recovered another **902** for which standard FID returned no hash. On Windows the corresponding counts were **1,678** and **1,370**. ARM32 had no FID-hash loss and only eight non-exact truth mappings.

This does not necessarily reflect platform quality. ARM32 imported embedded DWARF; the Windows logs show that the external PDB was not found; the macOS executables had no embedded DWARF and relied on function-start analysis. Symbol availability therefore explains part of the gap.

The discovery stage is partly configurable. Function-start analysis was enabled, while the aggressive instruction finders and searches inside data blocks were disabled. Part 2 tests those two settings directly. Standard FID also declines to hash functions below its four-code-unit minimum; changing that limit would require custom behaviour and a separate collision test.

### Main findings

- Identical rebuilds retained **100% of hashes once hashable**, but the end-to-end recoverability ceiling was only **70.72% on macOS**, **99.98% on ARM32**, and **86.06% on Windows**.
- On the same ARM32 target, changing **GCC 14.2 to Clang 22 retained 0 of 15,532 comparable full hashes**.
- Changing generic ARMv7 to Cortex-A7 retained **11.13% full** and **10.88% specific** hashes.
- Optimisation, frame-pointer policy, stack protection, PAC/BTI and AddressSanitizer produced substantial FID churn; the size of the effect varied by route.
- The old strip result concealed pre-hash loss: ARM full-hash survival was **97.59% among hashable pairs**, while only **59.83%** of tracked functions were hashable in both analyses.

## Part 2 — Minimising Ghidra recovery loss

In [3]:
route_names = {
    "macos-arm64": "macOS ARM64",
    "arm32-elf": "ARM32 ELF",
    "windows-x86-64": "Windows x86-64",
}

profile_order = [
    "default",
    "maximal-built-in",
    "shared-return-permissive",
    "call-target-recovery",
    "isolated-entry-recovery",
    "maximal-plus-isolated",
]
profile_names = {
    "default": "Default",
    "maximal-built-in": "Maximal built-in search",
    "shared-return-permissive": "Permissive shared returns",
    "call-target-recovery": "Direct-call recovery",
    "isolated-entry-recovery": "Isolated-flow recovery",
    "maximal-plus-isolated": "Maximal + isolated-flow",
}
all_rows = recovery[recovery["cohort"] == "all"]
for route in route_names:
    route_rows = (
        all_rows[all_rows["route"] == route]
        .set_index("profile")
        .loc[profile_order]
        .reset_index()
    )
    comparison = []
    for _, row in route_rows.iterrows():
        recovered = int(row["known_truth_recovered"])
        truth = int(row["known_truth_functions"])
        hashable = int(row["known_truth_hashable"])
        added = int(row["candidate_starts_added_vs_default"])
        confirmed = int(row["added_starts_confirmed_by_reference"])
        name = profile_names[row["profile"]]
        if row["selected_for_route"]:
            name += " — selected"
        comparison.append(
            {
                "Setting": name,
                "All functions Ghidra reported": f'{int(row["all_candidate_starts"]):,}',
                "Verified library functions found": (
                    f"{recovered:,}/{truth:,} ({100 * recovered / truth:.2f}%)"
                ),
                "Verified functions with a usable FID hash": (
                    f"{hashable:,}/{truth:,} ({100 * hashable / truth:.2f}%)"
                ),
                "Verified functions without a usable FID hash": (
                    f"{recovered - hashable:,}"
                ),
                "Extra verified functions vs default": f"{confirmed:,}",
                "Extra possible functions vs default (unverified)": (
                    f"{added - confirmed:,}"
                ),
                "Verified functions lost vs default": (
                    f'{int(row["known_truth_lost_vs_default"]):,}'
                ),
                "Functions assigned to the wrong library": (
                    f'{int(row["wrong_library_results"]):,}'
                ),
            }
        )

    display(Markdown(f"#### {route_names[route]}"))
    display(
        pd.DataFrame(comparison)
        .style.hide(axis="index")
        .set_properties(**{"text-align": "left", "white-space": "normal"})
        .set_table_styles(
            [
                {"selector": "th", "props": [("text-align", "left")]},
                {"selector": "table", "props": [("font-size", "12px")]},
            ]
        )
    )

#### macOS ARM64

Setting,All functions Ghidra reported,Verified library functions found,Verified functions with a usable FID hash,Verified functions without a usable FID hash,Extra verified functions vs default,Extra possible functions vs default (unverified),Verified functions lost vs default,Functions assigned to the wrong library
Default — selected,"3,120","2,446/2,821 (86.71%)","1,995/2,821 (70.72%)",451,0,0,0,0
Maximal built-in search,"3,120","2,446/2,821 (86.71%)","1,995/2,821 (70.72%)",451,0,0,0,0
Permissive shared returns,"3,120","2,446/2,821 (86.71%)","1,995/2,821 (70.72%)",451,0,0,0,0
Direct-call recovery,"3,120","2,446/2,821 (86.71%)","1,995/2,821 (70.72%)",451,0,0,0,0
Isolated-flow recovery,"3,121","2,446/2,821 (86.71%)","1,995/2,821 (70.72%)",451,0,1,0,0
Maximal + isolated-flow,"3,131","2,446/2,821 (86.71%)","1,995/2,821 (70.72%)",451,0,11,0,0


#### ARM32 ELF

Setting,All functions Ghidra reported,Verified library functions found,Verified functions with a usable FID hash,Verified functions without a usable FID hash,Extra verified functions vs default,Extra possible functions vs default (unverified),Verified functions lost vs default,Functions assigned to the wrong library
Default,"10,697","9,594/16,032 (59.84%)","9,594/16,032 (59.84%)",0,0,0,0,0
Maximal built-in search,"12,088","10,964/16,032 (68.39%)","10,964/16,032 (68.39%)",0,"1,391",0,0,0
Permissive shared returns,"10,697","9,594/16,032 (59.84%)","9,594/16,032 (59.84%)",0,0,0,0,0
Direct-call recovery,"10,697","9,594/16,032 (59.84%)","9,594/16,032 (59.84%)",0,0,0,0,0
Isolated-flow recovery,"11,659","10,472/16,032 (65.32%)","10,472/16,032 (65.32%)",0,929,33,0,0
Maximal + isolated-flow — selected,"16,549","15,213/16,032 (94.89%)","15,213/16,032 (94.89%)",0,"5,738",114,0,0


#### Windows x86-64

Setting,All functions Ghidra reported,Verified library functions found,Verified functions with a usable FID hash,Verified functions without a usable FID hash,Extra verified functions vs default,Extra possible functions vs default (unverified),Verified functions lost vs default,Functions assigned to the wrong library
Default,"10,563","10,090/10,929 (92.32%)","9,405/10,929 (86.06%)",685,0,0,0,0
Maximal built-in search,"10,563","10,090/10,929 (92.32%)","9,405/10,929 (86.06%)",685,0,0,0,0
Permissive shared returns,"10,534","10,067/10,929 (92.11%)","9,392/10,929 (85.94%)",675,0,0,23,0
Direct-call recovery,"10,563","10,090/10,929 (92.32%)","9,405/10,929 (86.06%)",685,0,0,0,0
Isolated-flow recovery — selected,"11,121","10,631/10,929 (97.27%)","9,546/10,929 (87.35%)","1,085",541,17,0,0
Maximal + isolated-flow,"11,126","10,631/10,929 (97.27%)","9,546/10,929 (87.35%)","1,085",541,22,0,0


### Reading the sensitivity tables

- **All functions Ghidra reported:** library, driver and runtime code.
- **Verified library functions:** functions independently mapped from the build evidence; this is the reliable recovery denominator.
- **Usable FID hash:** standard Ghidra FID produced a searchable hash. A verified function without one is not necessarily garbled; it may simply be too short or otherwise excluded by FID.
- **Extra verified functions:** improvements over Ghidra's default setting that agree with the independent truth data.
- **Extra possible functions:** false-positive risk. They may be real, but neither reference confirms them.
- **Assigned to the wrong library:** a verified function produced a unique FID match to a different library. Every tested recovery setting produced zero.

Libraries used:

- bzip2 1.0.7
- Expat R_2_7_5
- json-c 0.18
- miniz 3.1.1
- OpenSSL 3.5.7
- zlib 1.3.1